<style>
.jp-RenderedHTMLCommon table,
.rendered_html table {
  margin-left: 0 !important;
  margin-right: auto !important;
}
.jp-RenderedHTMLCommon th,
.jp-RenderedHTMLCommon td,
.rendered_html th,
.rendered_html td {
  text-align: left !important;
}
</style>

# 03 - Data Inspection and Cleaning

<div style="background:#F7FAFC; border:1px solid #D9E2EC; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/goal.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Notebook Header</b><br>
<b>Day:</b> Day 1 - EDA and preprocessing foundations<br>
<b>Difficulty:</b> Beginner<br>
<b>Estimated time:</b> 60-75 minutes<br>
<b>Prerequisites:</b> Notebook 02 completed (you should be comfortable creating DataFrames, selecting columns, and filtering rows)<br>
<b>Output:</b> You will inspect a messy table and create a cleaned version. Saves <code>data/customer_activity_raw.csv</code> and <code>data/customer_activity_clean.csv</code>.<br>
<b>Next notebook:</b> Basic statistics for data understanding
</div>

<details>
<summary><b>Instructor talk track</b></summary>

This notebook teaches the discipline of looking carefully before changing data. Encourage learners to describe the problem first, then fix it. Data cleaning is not random editing; every cleaning decision should have a reason.

</details>

## <img src="../../../assets/icons/map.svg" width="22" style="vertical-align:middle; margin-right:6px;"> How to Use This Notebook

For every cleaning step, use this rhythm:

1. Detect the problem.
2. Explain why it matters.
3. Apply one cleaning action.
4. Verify the result.
5. Document the decision in plain English.

<div style="background:#FFF8E1; border-left:6px solid #F2C94C; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Trainer Note</b><br>
Do not let learners rush through cleaning code. Ask what problem was detected, what action was taken, and how the result was verified.
</div>

## <img src="../../../assets/icons/map.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Learning Map

By the end of this notebook, you should be able to:

- Create a small messy dataset
- Inspect rows, columns, and data types
- Find missing values
- Find duplicate rows
- Clean text values
- Correct simple data type issues
- Handle missing numeric and categorical values
- Save a cleaned dataset

## <img src="../../../assets/icons/loop.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Signature Learning Loop

```text
QUESTION -> DATA -> CODE -> EVIDENCE -> DECISION
```

## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 1: Why Data Cleaning Matters

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
Data cleaning means finding and fixing data problems before analysis.
</div>

### Why this matters

Real-world data is rarely ready for analysis immediately.

If we analyze messy data, we may create:

- Wrong totals
- Wrong averages
- Misleading charts
- Incorrect group comparisons
- Poor decisions

### Common data problems

- Missing values
- Duplicate rows
- Extra spaces in text
- Inconsistent capitalization
- Wrong data types
- Impossible or unusual values

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Import the libraries needed for this notebook.
</div>

<div style="background:#FFF8E1; border-left:6px solid #F2C94C; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Read the Output</b><br>
This cell may not print anything. No error means the libraries loaded successfully.
</div>


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 2: Create a Messy Dataset

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
A messy dataset helps us practice realistic cleaning steps on a small table.
</div>

### Dataset story

The table contains customer purchase activity from a small retail business.

Some values are intentionally messy so we can practice cleaning.

### Problems intentionally included

- One duplicate customer row
- Missing `city`
- Missing `membership`
- Missing `monthly_spend`
- Missing `visits_per_month`
- Inconsistent text values such as `delhi`, `Delhi `, and ` Mumbai`
- An invalid negative spend value

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Create a small messy DataFrame.
</div>

<div style="background:#FFF8E1; border-left:6px solid #F2C94C; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Read the Output</b><br>
Do not clean yet. First look at the table and try to spot at least three problems manually.
</div>


In [2]:
raw_customers = pd.DataFrame({
    "customer_id": [101, 102, 103, 104, 105, 106, 106, 107, 108, 109],
    "name": ["Asha", "Ravi", "Meera", "John", "Fatima", "Chen", "Chen", "Sara", "Vikram", "Nina"],
    "city": [" Mumbai", "delhi", "Chennai", "Mumbai", np.nan, "chennai", "chennai", "Delhi ", "Mumbai", "Delhi"],
    "membership": ["Gold", "Silver", "Gold", "Bronze", "Silver", "gold", "gold", np.nan, "Bronze", "Silver"],
    "monthly_spend": [1200, 850, 1450, 700, np.nan, 1600, 1600, 950, -200, 1100],
    "visits_per_month": [4, 3, 5, 2, 6, 5, 5, np.nan, 1, 4],
    "signup_month": ["Jan", "Feb", "Feb", "Mar", "Apr", "Apr", "Apr", "May", "Jun", "Jun"],
})

raw_customers


,customer_id,name,city,membership,monthly_spend,visits_per_month,signup_month
0,101,Asha,Mumbai,Gold,1200.0,4.0,Jan
1,102,Ravi,delhi,Silver,850.0,3.0,Feb
2,103,Meera,Chennai,Gold,1450.0,5.0,Feb
3,104,John,Mumbai,Bronze,700.0,2.0,Mar
4,105,Fatima,NaN,Silver,NaN,6.0,Apr
5,106,Chen,chennai,gold,1600.0,5.0,Apr
6,106,Chen,chennai,gold,1600.0,5.0,Apr
7,107,Sara,Delhi,NaN,950.0,NaN,May
8,108,Vikram,Mumbai,Bronze,-200.0,1.0,Jun
9,109,Nina,Delhi,Silver,1100.0,4.0,Jun


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 3: First Inspection

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
Inspection means looking at the dataset before changing it.
</div>

### Why this matters

Inspection protects us from cleaning blindly.

Before changing data, we should know:

- How many records exist
- Which columns exist
- What the first rows look like
- Whether column names are understandable

### Useful first checks

- `head()` shows first rows
- `shape` shows rows and columns
- `columns` shows column names

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Run the first inspection commands.
</div>

<div style="background:#FFF8E1; border-left:6px solid #F2C94C; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Read the Output</b><br>
`shape` gives rows first and columns second. `head()` lets you visually inspect the first records.
</div>


In [3]:
print("Rows and columns:", raw_customers.shape)
print("Column names:", list(raw_customers.columns))

raw_customers.head()


Rows and columns: (10, 7)
Column names: ['customer_id', 'name', 'city', 'membership', 'monthly_spend', 'visits_per_month', 'signup_month']


,customer_id,name,city,membership,monthly_spend,visits_per_month,signup_month
0,101,Asha,Mumbai,Gold,1200.0,4.0,Jan
1,102,Ravi,delhi,Silver,850.0,3.0,Feb
2,103,Meera,Chennai,Gold,1450.0,5.0,Feb
3,104,John,Mumbai,Bronze,700.0,2.0,Mar
4,105,Fatima,NaN,Silver,NaN,6.0,Apr


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 4: Data Types

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
A data type tells Pandas what kind of values a column contains.
</div>

### Why this matters

Pandas behavior depends on data type.

Examples:

- Numeric columns can be averaged.
- Text columns can be cleaned with string methods.
- Date columns need date-aware operations.

### Common Pandas data types

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
  <thead>
    <tr>
      <th style="text-align:left; padding:6px 12px;">Data type</th>
      <th style="text-align:left; padding:6px 12px;">Simple meaning</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left; padding:6px 12px;"><code>int</code></td><td style="text-align:left; padding:6px 12px;">Whole numbers</td></tr>
    <tr><td style="text-align:left; padding:6px 12px;"><code>float</code></td><td style="text-align:left; padding:6px 12px;">Decimal numbers or numbers with missing values</td></tr>
    <tr><td style="text-align:left; padding:6px 12px;"><code>object</code></td><td style="text-align:left; padding:6px 12px;">Usually text or mixed values</td></tr>
  </tbody>
</table>

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Use `info()` to inspect column types and non-missing counts.
</div>

<div style="background:#FFF8E1; border-left:6px solid #F2C94C; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Read the Output</b><br>
Look for two things: each column's data type and each column's non-null count.
</div>


In [4]:
raw_customers.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       10 non-null     int64  
 1   name              10 non-null     object 
 2   city              9 non-null      object 
 3   membership        9 non-null      object 
 4   monthly_spend     9 non-null      float64
 5   visits_per_month  9 non-null      float64
 6   signup_month      10 non-null     object 
dtypes: float64(2), int64(1), object(4)
memory usage: 692.0+ bytes


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 5: Missing Values

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
A missing value means the value is not available in the dataset.
</div>

### Why missing values matter

- They can affect calculations.
- They can hide important data collection issues.
- They may mean different things in different columns.
- They should be counted before being filled or removed.

### Important caution

Do not fill missing values blindly.

First ask:

- Which column is missing?
- How many values are missing?
- Is the column numeric or categorical?
- Is missingness itself meaningful?

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Count missing values in each column.
</div>

<div style="background:#FFF8E1; border-left:6px solid #F2C94C; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Read the Output</b><br>
Columns with `0` have no missing values. Columns with values above `0` need a cleaning decision.
</div>


In [5]:
missing_counts = raw_customers.isna().sum()

missing_counts


customer_id         0
name                0
city                1
membership          1
monthly_spend       1
visits_per_month    1
signup_month        0
dtype: int64

## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 6: Duplicate Rows

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
A duplicate row is repeated data that may accidentally count the same record more than once.
</div>

### Why duplicates matter

- They can inflate totals.
- They can distort averages.
- They can make the dataset look larger than it really is.
- They can cause repeated records in reports.

### What `duplicated()` does

`duplicated()` marks rows that are repeated after their first occurrence.

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Find duplicate rows.
</div>

<div style="background:#FFF8E1; border-left:6px solid #F2C94C; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Read the Output</b><br>
Any row shown here is a duplicate copy. If no rows are shown, no full duplicate rows were found.
</div>


In [6]:
duplicate_rows = raw_customers[raw_customers.duplicated()]

duplicate_rows


,customer_id,name,city,membership,monthly_spend,visits_per_month,signup_month
6,106,Chen,chennai,gold,1600.0,5.0,Apr


## <img src="../../../assets/icons/practice.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Checkpoint 1: Inspect Before Cleaning

Answer these before continuing:

1. How many rows and columns are in the raw dataset?
2. Which columns have missing values?
3. Is there a duplicate row?
4. Which text columns look inconsistent?

<div style="background:#FFF8E1; border-left:6px solid #F2C94C; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Expected thinking</b><br>
Do not clean immediately. First list the problems clearly.
</div>


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 7: Create a Cleaning Copy

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
Create a copy before cleaning so the raw data remains unchanged.
</div>

### Why this matters

A cleaning copy lets us experiment safely.

If a cleaning step goes wrong, the original raw data is still available.

<div style="background:#FFECEC; border-left:6px solid #EB5757; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/caution.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Caution</b><br>
Do not overwrite raw data until you are confident your cleaning steps are correct.
</div>

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Create a copy of the raw dataset.
</div>

<div style="background:#FFF8E1; border-left:6px solid #F2C94C; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Read the Output</b><br>
The raw data and cleaning copy should start with the same shape.
</div>


In [7]:
clean_customers = raw_customers.copy()

print("Raw data shape:", raw_customers.shape)
print("Cleaning copy shape:", clean_customers.shape)


Raw data shape: (10, 7)
Cleaning copy shape: (10, 7)


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 8: Clean Text Values

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
Text values should be consistent before analysis.
</div>

### Why this matters

To humans, `delhi`, `Delhi`, and `Delhi ` may look like the same city.

To software, they are different values unless we clean them.

### Cleaning text often includes

- Removing extra spaces
- Standardizing capitalization
- Fixing repeated spellings

### Methods used here

- `str.strip()` removes extra spaces at the beginning and end
- `str.title()` standardizes capitalization

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Clean the `city` and `membership` text columns.
</div>

<div style="background:#FFF8E1; border-left:6px solid #F2C94C; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Read the Output</b><br>
Check that city and membership values now follow a consistent text format.
</div>


In [16]:
clean_customers["city"] = clean_customers["city"].str.strip().str.title()
clean_customers["membership"] = clean_customers["membership"].str.strip().str.title()

clean_customers[["city", "membership"]]


,city,membership
0,Mumbai,Gold
1,Delhi,Silver
2,Chennai,Gold
3,Mumbai,Bronze
4,NaN,Silver
5,Chennai,Gold
7,Delhi,NaN
8,Mumbai,Bronze
9,Delhi,Silver


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 9: Remove Duplicates

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
Removing duplicate rows prevents repeated records from affecting summaries.
</div>

### Why this matters

If the same row appears twice, the customer may be counted twice.

This can affect:

- Row counts
- Totals
- Averages
- Group summaries

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Drop duplicate rows and compare row counts.
</div>

<div style="background:#FFF8E1; border-left:6px solid #F2C94C; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Read the Output</b><br>
Rows removed should equal rows before minus rows after. This confirms how many duplicate records were removed.
</div>


In [17]:
rows_before = clean_customers.shape[0]
clean_customers = clean_customers.drop_duplicates()
rows_after = clean_customers.shape[0]

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Rows removed:", rows_before - rows_after)


Rows before: 9
Rows after: 9
Rows removed: 0


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 10: Handle Missing Numeric Values

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
Numeric missing values can be filled using a reasonable summary value.
</div>

### Why this matters

Many calculations and later workflows cannot handle missing numeric values directly.

But the fill value must be chosen carefully.

### Common beginner approach

- Use median for spending-like columns.
- Use median when values may contain unusual highs or lows.
- Document which value was used.

<div style="background:#FFECEC; border-left:6px solid #EB5757; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/caution.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Caution</b><br>
Filling missing values changes the dataset. Always explain the reason.
</div>

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Fill missing `monthly_spend` and `visits_per_month` values using median values.
</div>

<div style="background:#FFF8E1; border-left:6px solid #F2C94C; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Read the Output</b><br>
Record the median values used. These values become part of the cleaning documentation.
</div>


In [18]:
spend_median = clean_customers["monthly_spend"].median()
visits_median = clean_customers["visits_per_month"].median()

clean_customers["monthly_spend"] = clean_customers["monthly_spend"].fillna(spend_median)
clean_customers["visits_per_month"] = clean_customers["visits_per_month"].fillna(visits_median)

print("Spend median used:", spend_median)
print("Visits median used:", visits_median)


Spend median used: 1025.0
Visits median used: 4.0


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 11: Handle Missing Categorical Values

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
Categorical missing values can be filled with a clear label such as `Unknown`.
</div>

### Why use `Unknown`?

- It does not pretend we know the real value.
- It keeps the row in the dataset.
- It makes missing categories visible during analysis.
- It is safer than guessing a city or membership group.

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Fill missing `city` and `membership` values with `Unknown`.
</div>

<div style="background:#FFF8E1; border-left:6px solid #F2C94C; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Read the Output</b><br>
Look for `Unknown` values. They show where the original data did not provide a category.
</div>


In [19]:
clean_customers["city"] = clean_customers["city"].fillna("Unknown")
clean_customers["membership"] = clean_customers["membership"].fillna("Unknown")

clean_customers[["city", "membership"]]


,city,membership
0,Mumbai,Gold
1,Delhi,Silver
2,Chennai,Gold
3,Mumbai,Bronze
4,Unknown,Silver
5,Chennai,Gold
7,Delhi,Unknown
8,Mumbai,Bronze
9,Delhi,Silver


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 12: Handle Unusual Values

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
An unusual value is a value that does not make sense for the column.
</div>

### Why this matters

Unusual values can strongly affect summaries.

Example:

- Monthly spend should not be negative.
- A value like `-200` should be reviewed and corrected.

### Professional habit

Do not delete or replace unusual values silently.

First:

1. Detect the row.
2. Display the row.
3. Decide what to do.
4. Document the decision.

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Find negative monthly spend values and replace them with the median.
</div>

<div style="background:#FFF8E1; border-left:6px solid #F2C94C; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Read the Output</b><br>
First inspect the negative row. Then check the `describe()` output after replacement.
</div>


In [12]:
negative_spend_rows = clean_customers[clean_customers["monthly_spend"] < 0]
display(negative_spend_rows)

clean_customers.loc[clean_customers["monthly_spend"] < 0, "monthly_spend"] = spend_median

clean_customers["monthly_spend"].describe()


,customer_id,name,city,membership,monthly_spend,visits_per_month,signup_month
8,108,Vikram,Mumbai,Bronze,-200.0,1.0,Jun


count       9.000000
mean     1100.000000
std       282.566364
min       700.000000
25%       950.000000
50%      1025.000000
75%      1200.000000
max      1600.000000
Name: monthly_spend, dtype: float64

## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 13: Final Quality Check

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
After cleaning, run checks again to confirm the problems were fixed.
</div>

### Why this matters

Cleaning is not complete just because code ran.

We must verify that the expected problems are gone.

### Final checks

- Missing values
- Duplicate rows
- Negative spending values
- Cleaned text categories

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Run final quality checks.
</div>

<div style="background:#FFF8E1; border-left:6px solid #F2C94C; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Read the Output</b><br>
Missing values, duplicate rows, and negative spend rows should be zero after cleaning.
</div>


In [ ]:
print("Missing values:")
print(clean_customers.isna().sum())

print("\nDuplicate rows:", clean_customers.duplicated().sum())
print("Negative spend rows:", (clean_customers["monthly_spend"] < 0).sum())

display(clean_customers)


## <img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Concept 14: Save the Cleaned Dataset

<div style="background:#EAF3FF; border-left:6px solid #2F80ED; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/concept.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Concept</b><br>
A cleaned dataset should be saved separately from the raw dataset.
</div>

### Why this matters

Saving both raw and cleaned files gives traceability.

Good habit:

- Keep raw data unchanged.
- Save cleaned data with a clear filename.
- Use the cleaned file in later analysis.
- Re-run cleaning if raw data changes.

<div style="background:#E9F8EF; border-left:6px solid #27AE60; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/code.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Code Lab</b><br>
Save raw and cleaned customer datasets into the project `data` folder.
</div>

<div style="background:#FFF8E1; border-left:6px solid #F2C94C; padding:14px; border-radius:6px;">
<img src="../../../assets/icons/read-output.svg" width="22" style="vertical-align:middle; margin-right:6px;"><b>Read the Output</b><br>
The printed paths confirm where the raw and cleaned CSV files were saved.
</div>


In [20]:
current = Path.cwd().resolve()
project_root = current

for candidate in [current, *current.parents]:
    if candidate.name == "applied_ds_ml":
        project_root = candidate
        break

data_dir = project_root / "data"
data_dir.mkdir(parents=True, exist_ok=True)

raw_path = data_dir / "customer_activity_raw.csv"
clean_path = data_dir / "customer_activity_clean.csv"

raw_customers.to_csv(raw_path, index=False)
clean_customers.to_csv(clean_path, index=False)

print("Raw dataset saved to:", raw_path)
print("Clean dataset saved to:", clean_path)


Raw dataset saved to: C:\Users\nagcl\lab\applied_ds_ml\data\customer_activity_raw.csv
Clean dataset saved to: C:\Users\nagcl\lab\applied_ds_ml\data\customer_activity_clean.csv


## <img src="../../../assets/icons/recap.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Cleaning Summary Table

<table style="margin-left:0; margin-right:auto; border-collapse:collapse; text-align:left;">
  <thead>
    <tr>
      <th style="text-align:left; padding:6px 12px;">Problem</th>
      <th style="text-align:left; padding:6px 12px;">How we detected it</th>
      <th style="text-align:left; padding:6px 12px;">How we handled it</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:left; padding:6px 12px;">Missing numeric values</td><td style="text-align:left; padding:6px 12px;"><code>isna().sum()</code></td><td style="text-align:left; padding:6px 12px;">Filled with median</td></tr>
    <tr><td style="text-align:left; padding:6px 12px;">Missing text values</td><td style="text-align:left; padding:6px 12px;"><code>isna().sum()</code></td><td style="text-align:left; padding:6px 12px;">Filled with <code>Unknown</code></td></tr>
    <tr><td style="text-align:left; padding:6px 12px;">Duplicate rows</td><td style="text-align:left; padding:6px 12px;"><code>duplicated()</code></td><td style="text-align:left; padding:6px 12px;">Removed duplicates</td></tr>
    <tr><td style="text-align:left; padding:6px 12px;">Messy text</td><td style="text-align:left; padding:6px 12px;">Manual inspection</td><td style="text-align:left; padding:6px 12px;">Used strip and title case</td></tr>
    <tr><td style="text-align:left; padding:6px 12px;">Negative spending</td><td style="text-align:left; padding:6px 12px;">Condition check</td><td style="text-align:left; padding:6px 12px;">Replaced with median</td></tr>
  </tbody>
</table>


## <img src="../../../assets/icons/caution.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Common Mistakes to Avoid

<div style="background:#FFECEC; border-left:6px solid #EB5757; padding:14px; border-radius:6px;">
<b>Read this before practice.</b>
</div>

- Cleaning data before inspecting it.
- Overwriting raw data too early.
- Filling missing values without explaining the reason.
- Removing rows without checking how many rows are removed.
- Ignoring text inconsistencies like `delhi`, `Delhi`, and `Delhi `.
- Assuming every unusual value is wrong without checking context.
- Forgetting to run final quality checks after cleaning.

## <img src="../../../assets/icons/recap.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Cleaning Checklist

Before calling a dataset clean, confirm:

- Row count changes are understood.
- Missing values are handled or intentionally kept.
- Duplicate rows are reviewed.
- Text categories are standardized.
- Invalid numeric values are reviewed.
- Cleaning decisions are documented.
- Raw and cleaned files are saved separately.


## <img src="../../../assets/icons/practice.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Practice Task

Complete these tasks:

1. Add one new row with a missing `membership` value.
2. Count missing values again.
3. Fill the missing `membership` value with `Unknown`.
4. Add one row with extra spaces in `city`, such as `" Mumbai "`.
5. Clean the city column again using `str.strip().str.title()`.
6. Write three observations about the cleaned dataset.

## <img src="../../../assets/icons/practice.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Exit Ticket

Answer these before closing the notebook:

1. Why should we inspect data before cleaning it?
2. What does `isna().sum()` show?
3. Why do we create a copy before cleaning?
4. Why is `Unknown` sometimes better than guessing a category?
5. Why should cleaned data be saved separately from raw data?

## <img src="../../../assets/icons/recap.svg" width="22" style="vertical-align:middle; margin-right:6px;"> Ready for the Next Notebook

You are ready for the next notebook if you can:

- Inspect rows, columns, and data types.
- Count missing values.
- Detect duplicate rows.
- Clean basic text inconsistencies.
- Fill missing numeric and categorical values.
- Save a cleaned dataset.
